In [0]:
-- A few questions need answering first
-- 1. How many CKD-staged conditions exist and how many have staging at all?
SELECT
  ckd_stage,
  condition_desc,
  COUNT(*) AS records,
  COUNT(DISTINCT patient_id) AS patients
FROM health_lakehouse.gold.fact_conditions
WHERE ckd_stage IS NOT NULL
GROUP BY ckd_stage, condition_desc
ORDER BY ckd_stage;

-- 2. Do patients actually progress through multiple stages, or sit at one?
WITH patient_stages AS (
  SELECT patient_id,COUNT(DISTINCT ckd_stage) AS n_stages
  FROM health_lakehouse.gold.fact_conditions
  WHERE ckd_stage IS NOT NULL
  GROUP BY patient_id
)
SELECT n_stages,COUNT(*) AS patients
FROM patient_stages
GROUP BY n_stages
ORDER BY n_stages;

-- 3. What does one multi-stage patient's actual timeline look like?
WITH multi AS (
  SELECT patient_id
  FROM health_lakehouse.gold.fact_conditions
  WHERE ckd_stage IS NOT NULL
  GROUP BY patient_id
  HAVING COUNT(DISTINCT ckd_stage) >= 3
  LIMIT 1
)
SELECT c.patient_id, c.ckd_stage, c.condition_desc, c.onset_date, c.resolved_date
FROM health_lakehouse.gold.fact_conditions c
JOIN multi USING (patient_id)
WHERE c.ckd_stage IS NOT NULL
ORDER BY c.onset_date;

In [0]:
-- ============================================================
-- CKD PROGRESSION ANALYSIS
-- Tracks each patient's journey through CKD stages over time.
-- Core technique: LAG/LEAD over a per-patient, date-ordered
-- window to detect stage-to-stage transitions and measure how
-- long patients dwell at each stage before progressing.
-- ============================================================
WITH stage_events AS (
  -- first onset per patient per stage (a stage can be re-coded;
  -- we take the earliest date a patient reached each stage)
  SELECT
    patient_id,
    ckd_stage,
    MIN(onset_date) AS stage_onset
  FROM health_lakehouse.gold.fact_conditions
  WHERE ckd_stage IS NOT NULL
  GROUP BY patient_id, ckd_stage
),
sequenced AS (
  SELECT
    patient_id,
    ckd_stage,
    stage_onset,
    -- the next stage this patient reaches, and when
    LEAD(ckd_stage) OVER (PARTITION BY patient_id ORDER BY stage_onset) AS next_stage,
    LEAD(stage_onset) OVER (PARTITION BY patient_id ORDER BY stage_onset) AS next_onset,
    -- position in this patient's journey
    ROW_NUMBER() OVER (PARTITION BY patient_id ORDER BY stage_onset) AS journey_step
  FROM stage_events
)
SELECT
  ckd_stage,
  next_stage,
  COUNT(*) AS transitions,
  ROUND(AVG(DATEDIFF(next_onset,stage_onset) / 365.25), 1) AS avg_years_in_stage,
  ROUND(MIN(DATEDIFF(next_onset,stage_onset) / 365.25), 1) AS min_years,
  ROUND(MAX(DATEDIFF(next_onset,stage_onset) / 365.25), 1) AS max_years
FROM sequenced
WHERE next_stage IS NOT NULL
GROUP BY ckd_stage, next_stage
ORDER BY ckd_stage, next_stage;

In [0]:
-- Investigate the backward transitions: do these patients have transplant events? 4-> 1 and 5->1 (for the 3 patients)
WITH backward_patients AS (
  SELECT DISTINCT patient_id
  FROM (
    SELECT patient_id, ckd_stage,
           LEAD(ckd_stage) OVER (PARTITION BY patient_id ORDER BY MIN(onset_date)) AS next_stage
    FROM health_lakehouse.gold.fact_conditions
    WHERE ckd_stage IS NOT NULL
    GROUP BY patient_id, ckd_stage
  )
  WHERE next_stage < ckd_stage -- stage went DOWN
)
SELECT c.patient_id, c.condition_desc, c.onset_date
FROM health_lakehouse.gold.fact_conditions c
JOIN backward_patients b USING (patient_id)
WHERE LOWER(c.condition_desc) RLIKE 'transplant|kidney|renal'
ORDER BY c.patient_id, c.onset_date;

In [0]:
-- Most common progression pathways — concatenate each patient's
-- ordered stage sequence into a single trajectory string.
WITH stage_events AS (
  SELECT patient_id, ckd_stage, MIN(onset_date) AS stage_onset
  FROM health_lakehouse.gold.fact_conditions
  WHERE ckd_stage IS NOT NULL
  GROUP BY patient_id, ckd_stage
),
journeys AS (
  SELECT
    patient_id,
    -- ordered array of stages → readable path like "1 → 2 → 3 → 5"
    array_join(
      transform(
        array_sort(collect_list(struct(stage_onset, ckd_stage))),
        x -> CAST(x.ckd_stage AS STRING)
      ),
      ' → '
    ) AS pathway,
    COUNT(*) AS n_stages,
    MIN(stage_onset) AS journey_start,
    MAX(stage_onset) AS journey_end
  FROM stage_events
  GROUP BY patient_id
)
SELECT
  pathway,
  COUNT(*) AS patients,
  ROUND(AVG(DATEDIFF(journey_end, journey_start) / 365.25), 1) AS avg_journey_years
FROM journeys
GROUP BY pathway
ORDER BY patients DESC
LIMIT 20;

In [0]:
-- ============================================================
-- CKD PROGRESSION — clinical validation via eGFR
-- CKD stages are DEFINED by eGFR (mL/min/1.73m²):
-- Stage 1 ≥90 | Stage 2 60-89 | Stage 3 30-59 | Stage 4 15-29 | Stage 5 <15
-- If the coded stages are meaningful, mean eGFR should fall
-- monotonically as stage rises. This joins each staged patient's
-- eGFR readings (taken while at that stage) to their stage code.
-- ============================================================
WITH stage_windows AS (
  -- each patient's stage and the date range they were at it
  SELECT
    patient_id,
    ckd_stage,
    MIN(onset_date) AS stage_start,
    LEAD(MIN(onset_date)) OVER (
      PARTITION BY patient_id ORDER BY MIN(onset_date)
    ) AS stage_end
  FROM health_lakehouse.gold.fact_conditions
  WHERE ckd_stage IS NOT NULL
  GROUP BY patient_id, ckd_stage
),
egfr AS (
  SELECT patient_id, DATE(observed_ts) AS obs_date, value_numeric AS egfr
  FROM health_lakehouse.gold.fact_observations
  WHERE measure_group = 'lab: egfr'
),
matched AS (
  -- attribute each eGFR reading to the stage the patient was in at the time
  SELECT
    sw.ckd_stage,
    e.egfr
  FROM stage_windows sw
  JOIN egfr e
    ON e.patient_id = sw.patient_id
   AND e.obs_date  >= sw.stage_start
   AND (e.obs_date <  sw.stage_end OR sw.stage_end IS NULL)
)
SELECT
  ckd_stage,
  COUNT(*) AS egfr_readings,
  ROUND(AVG(egfr), 1) AS avg_egfr,
  ROUND(PERCENTILE(egfr, 0.5), 1) AS median_egfr,
  ROUND(MIN(egfr), 1) AS min_egfr,
  ROUND(MAX(egfr), 1) AS max_egfr
FROM matched
GROUP BY ckd_stage
ORDER BY ckd_stage;